In [27]:
import os
import logging
import argparse
import pandas as pd
from dotenv import load_dotenv
from neo4j_handler import Neo4jHandler

# Import Splink components using the confirmed syntax
import splink.comparison_library as cl
from splink import DuckDBAPI, Linker, SettingsCreator, block_on, splink_datasets

# Configure logging.
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

# Load environment variables from the .env file.
load_dotenv()
NEO4J_URI = os.getenv("NEO4J_URI")
NEO4J_USER = os.getenv("NEO4J_USER")
NEO4J_PASSWORD = os.getenv("NEO4J_PASSWORD")
if not (NEO4J_URI and NEO4J_USER and NEO4J_PASSWORD):
    raise ValueError("Please set NEO4J_URI, NEO4J_USER, and NEO4J_PASSWORD in your .env file")

handler = Neo4jHandler(NEO4J_URI, NEO4J_USER, NEO4J_PASSWORD)

logger.info("Initializing DuckDB backend for Splink...")
db_api = DuckDBAPI()  # Create a DuckDB backend instance
from splink.exploratory import profile_columns

INFO:neo4j_handler:Connected to Neo4j at bolt://localhost:7687 as user neo4j
INFO:__main__:Initializing DuckDB backend for Splink...


In [28]:
def fetch_identities(handler):
    """
    Fetch all Identity nodes from the Neo4j database and return them as a list of dictionaries.
    Expected columns: id, full_name, email_address, zip_code, phone_number.
    """
    query = """
    MATCH (i:Identity)
    RETURN distinct i
    """
    result = handler.execute_read(query)
    # Consume the result within the transaction scope.
    records = [record.data() for record in result]
    return records


identities = fetch_identities(handler)

In [29]:
data = [identity["i"] for identity in identities]  
identities_df = pd.DataFrame(data)


identities_df.head()



,full_name,email_address,wcc_component,last_name,phone_number,id,full_address,first_name,zip_code
0,Isaiah Romero,isaiah.romero@example.com,0,Romero,9398997994,c78a7d47-03d6-4b60-9052-ee14ecfd12c8,"2407 Daisy Dr, Norfolk, Missouri, United State...",Isaiah,NaN
1,Bill Barnes,bill.barnes@example.com,0,Barnes,7344373234,12d25465-9ecc-4d5f-9e3c-90c73833a039,"4272 Pockrus Page Rd, Forney, Idaho, United St...",Bill,51766
2,Bill Barnes,bill.barnes@example.com,0,Barnes,7344373234,24fb15fc-5084-40a2-b40b-10bf736fed21,"4272 Pockrus Page Rd, Forney, Idaho, United St...",Bill,NaN
3,Bill Barnes,bill.barnes@example.com,0,Barnes,NaN,2731b8da-841e-44a4-8cb6-e205ae2bcac3,"4272 Pockrus Page Rd, Forney, Idaho, United St...",Bill,NaN
4,Bill Barnes,bill.barnes@example.com,0,Barnes,7344373234,61b80148-f747-4538-812d-7772b3487011,"4272 Pockrus Page Rd, Forney, Idaho, United St...",Bill,51766


In [30]:
identities_df = identities_df.rename(columns={"id": "unique_id"})
identities_df.head()

,full_name,email_address,wcc_component,last_name,phone_number,unique_id,full_address,first_name,zip_code
0,Isaiah Romero,isaiah.romero@example.com,0,Romero,9398997994,c78a7d47-03d6-4b60-9052-ee14ecfd12c8,"2407 Daisy Dr, Norfolk, Missouri, United State...",Isaiah,NaN
1,Bill Barnes,bill.barnes@example.com,0,Barnes,7344373234,12d25465-9ecc-4d5f-9e3c-90c73833a039,"4272 Pockrus Page Rd, Forney, Idaho, United St...",Bill,51766
2,Bill Barnes,bill.barnes@example.com,0,Barnes,7344373234,24fb15fc-5084-40a2-b40b-10bf736fed21,"4272 Pockrus Page Rd, Forney, Idaho, United St...",Bill,NaN
3,Bill Barnes,bill.barnes@example.com,0,Barnes,NaN,2731b8da-841e-44a4-8cb6-e205ae2bcac3,"4272 Pockrus Page Rd, Forney, Idaho, United St...",Bill,NaN
4,Bill Barnes,bill.barnes@example.com,0,Barnes,7344373234,61b80148-f747-4538-812d-7772b3487011,"4272 Pockrus Page Rd, Forney, Idaho, United St...",Bill,51766


In [31]:


profile_columns(identities_df, db_api, column_expressions=["first_name", "last_name", "zip_code"])

alt.VConcatChart(...)

In [32]:
from splink.blocking_analysis import (
    cumulative_comparisons_to_be_scored_from_blocking_rules_chart,
)

blocking_rules = [
    block_on("wcc_component"),
    block_on("first_name"),
    block_on("last_name"),
    block_on("zip_code"),  
    
]


cumulative_comparisons_to_be_scored_from_blocking_rules_chart(
    table_or_tables=identities_df,
    blocking_rules=blocking_rules,
    db_api=db_api,
    link_type="dedupe_only",
)

alt.Chart(...)

In [33]:


# Add a source_dataset column to the DataFrame.
identities_df["source_dataset"] = "dataset_1"  # Assign a default value for all records.

logger.info(f"Fetched {len(identities_df)} identity records.")

# Create Splink settings using the new syntax.
# Here we use our existing columns:
# - NameComparison on full_name
# - EmailComparison on email_address
# - ExactMatch on zip_code and phone_number (with term frequency adjustments for zip_code)
settings = SettingsCreator(
    link_type="dedupe_only",
    comparisons=[
        # cl.NameComparison("full_name"),
        # cl.EmailComparison("email_address"),

        cl.ForenameSurnameComparison(
            "first_name",
            "last_name",
            forename_surname_concat_col_name="first_name_last_name_concat",
        ),
        cl.LevenshteinAtThresholds("email_address"),
        cl.ExactMatch("zip_code"),
        cl.ExactMatch("phone_number"),
        cl.LevenshteinAtThresholds("full_address")
    ],
    #     cl.LevenshteinAtThresholds("full_name").configure(term_frequency_adjustments=True),
    #     cl.LevenshteinAtThresholds("email_address").configure(term_frequency_adjustments=True),
    #     cl.LevenshteinAtThresholds("zip_code").configure(term_frequency_adjustments=True),
    #     cl.LevenshteinAtThresholds("phone_number").configure(term_frequency_adjustments=True),
    # ],
    retain_intermediate_calculation_columns=True,
    blocking_rules_to_generate_predictions=blocking_rules,
)


identities_df["first_name_last_name_concat"] = identities_df["first_name"] + " " + identities_df["last_name"]

logger.info("Initializing Splink Linker...")
# Initialize the generic Linker with the DataFrame, settings, and DuckDB backend.
linker = Linker(identities_df, settings, db_api=db_api)



INFO:__main__:Fetched 4672 identity records.
INFO:__main__:Initializing Splink Linker...


In [34]:
# -------------------------
# Training steps:
# Estimate the probability that two random records match.
linker.training.estimate_probability_two_random_records_match(
    [block_on("wcc_component")],
    recall=0.6,
)

INFO:splink.internals.linker_components.training:Probability two random records match is estimated to be  0.0416.
This means that amongst all possible pairwise record comparisons, one in 24.01 are expected to match.  With 10,911,456 total possible comparisons, we expect a total of around 454,366.67 matching pairs


In [35]:
# Estimate u probabilities using random sampling.
linker.training.estimate_u_using_random_sampling(max_pairs=1e8)


INFO:splink.internals.estimate_u:----- Estimating u probabilities using random sampling -----
INFO:splink.internals.m_u_records_to_parameters:u probability not trained for first_name_last_name - Match on reversed cols: first_name and last_name (both directions) (comparison vector value: 5). This usually means the comparison level was never observed in the training data.
INFO:splink.internals.estimate_u:
Estimated u probabilities using random sampling
INFO:splink.internals.settings:
Your model is not yet fully trained. Missing estimates for:
    - first_name_last_name (some u values are not trained, no m values are trained).
    - email_address (no m values are trained).
    - zip_code (no m values are trained).
    - phone_number (no m values are trained).
    - full_address (no m values are trained).


In [36]:

training_blocking_rule = block_on("wcc_component")
trainin_sesion_wcc= (
    linker.training.estimate_parameters_using_expectation_maximisation(
        training_blocking_rule, estimate_without_term_frequencies=True
    )
)

INFO:splink.internals.em_training_session:
----- Starting EM training session -----

INFO:splink.internals.em_training_session:Estimating the m probabilities of the model by blocking on:
l."wcc_component" = r."wcc_component"

Parameter estimates will be made for the following comparison(s):
    - first_name_last_name
    - email_address
    - zip_code
    - phone_number
    - full_address

Parameter estimates cannot be made for the following comparison(s) since they are used in the blocking rules: 
INFO:splink.internals.expectation_maximisation:
Level Match on reversed cols: first_name and last_name (both directions) on comparison first_name_last_name not observed in dataset, unable to train m value

INFO:splink.internals.expectation_maximisation:Iteration 1: Largest change in params was 0.0411 in probability_two_random_records_match
INFO:splink.internals.expectation_maximisation:Iteration 2: Largest change in params was 0.000585 in the m_probability of email_address, level `All other 

In [37]:
linker.visualisations.match_weights_chart()

alt.VConcatChart(...)

In [38]:
linker.evaluation.unlinkables_chart()

alt.LayerChart(...)

In [39]:
df_predict = linker.inference.predict()
df_e = df_predict.as_pandas_dataframe()
df_e = df_e[abs(df_e["match_probability"]) < .9]
df_e = df_e[abs(df_e["match_probability"]) > .5]

# df_e = df_e[df_e["match_weig"] > 0.1]

df_e.count()

INFO:splink.internals.linker_components.inference:Blocking time: 0.15 seconds
INFO:splink.internals.linker_components.inference:Predict time: 7.51 seconds
 -- WARNING --
You have called predict(), but there are some parameter estimates which have neither been estimated or specified in your settings dictionary.  To produce predictions the following untrained trained parameters will use default values.
Comparison: 'first_name_last_name':
    m values not fully trained
Comparison: 'first_name_last_name':
    u values not fully trained


match_weight                        14
match_probability                   14
unique_id_l                         14
unique_id_r                         14
first_name_l                        14
first_name_r                        14
last_name_l                         14
last_name_r                         14
first_name_last_name_concat_l       14
first_name_last_name_concat_r       14
gamma_first_name_last_name          14
tf_first_name_last_name_concat_l    14
tf_first_name_last_name_concat_r    14
tf_last_name_l                      14
tf_last_name_r                      14
tf_first_name_l                     14
tf_first_name_r                     14
bf_first_name_last_name             14
bf_tf_adj_first_name_last_name      14
email_address_l                     12
email_address_r                     11
gamma_email_address                 14
bf_email_address                    14
zip_code_l                           8
zip_code_r                          11
gamma_zip_code           

In [40]:
records_to_plot = df_e.to_dict(orient="records")
linker.visualisations.waterfall_chart(records_to_plot, filter_nulls=False)

alt.LayerChart(...)

In [41]:
clusters = linker.clustering.cluster_pairwise_predictions_at_threshold(
    df_predict, threshold_match_probability=0.95
)

INFO:splink.internals.connected_components:Completed iteration 1, num representatives needing updating: 0


In [42]:
df_clusters = clusters.as_pandas_dataframe()
df_clusters.head()

,cluster_id,full_name,email_address,wcc_component,last_name,phone_number,unique_id,full_address,first_name,zip_code,source_dataset,first_name_last_name_concat
0,4002cdff-d764-4250-9c1c-3dcc06c24e3f,Isaiah Romero,isaiah.romero@example.com,0,Romero,9398997994,c78a7d47-03d6-4b60-9052-ee14ecfd12c8,"2407 Daisy Dr, Norfolk, Missouri, United State...",Isaiah,None,dataset_1,Isaiah Romero
1,12d25465-9ecc-4d5f-9e3c-90c73833a039,Bill Barnes,bill.barnes@example.com,0,Barnes,7344373234,12d25465-9ecc-4d5f-9e3c-90c73833a039,"4272 Pockrus Page Rd, Forney, Idaho, United St...",Bill,51766,dataset_1,Bill Barnes
2,12d25465-9ecc-4d5f-9e3c-90c73833a039,Bill Barnes,bill.barnes@example.com,0,Barnes,7344373234,24fb15fc-5084-40a2-b40b-10bf736fed21,"4272 Pockrus Page Rd, Forney, Idaho, United St...",Bill,None,dataset_1,Bill Barnes
3,12d25465-9ecc-4d5f-9e3c-90c73833a039,Bill Barnes,bill.barnes@example.com,0,Barnes,None,2731b8da-841e-44a4-8cb6-e205ae2bcac3,"4272 Pockrus Page Rd, Forney, Idaho, United St...",Bill,None,dataset_1,Bill Barnes
4,12d25465-9ecc-4d5f-9e3c-90c73833a039,Bill Barnes,bill.barnes@example.com,0,Barnes,7344373234,61b80148-f747-4538-812d-7772b3487011,"4272 Pockrus Page Rd, Forney, Idaho, United St...",Bill,51766,dataset_1,Bill Barnes


In [43]:
# write the clusters to neo4j
rows = df_clusters.to_dict("records")

# Cypher query to update nodes: it unwinds each row and sets the cluster_id property.
update_query = """
UNWIND $rows as row
MATCH (n {id: row.id})
SET n.cluster_id = row.cluster_id
RETURN count(n) AS updated_count
"""

# Use your Neo4jHandler's execute_write method to run the update.
handler.execute_write(update_query, parameters={"rows": rows})